In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql.functions import max as spark_max

today = datetime.today().strftime("%Y-%m-%d")
end_forecast = (datetime.today() + timedelta(days=7)).strftime("%Y-%m-%d")

# Verificar última data guardada na Bronze histórica
try:
    df_existing = spark.read.format("delta").load("Files/bronze/weather_raw")
    last_date = df_existing.agg(spark_max("time")).collect()[0][0]
    # Partir do dia seguinte à última data guardada
    start_date = (datetime.strptime(last_date[:10], "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"Dados existentes até {last_date[:10]}, a pedir desde {start_date}")
except:
    # Se não existir Bronze, começa do início
    start_date = "2025-11-27"
    print(f"Bronze não existe, a pedir desde {start_date}")

# Só corre se houver dados novos para ir buscar
if start_date <= today:
    # Histórico
    response = requests.get(
        f"https://historical-forecast-api.open-meteo.com/v1/forecast?"
        f"latitude=40.66&longitude=-7.91"
        f"&start_date={start_date}&end_date={today}"
        f"&hourly=temperature_2m,relative_humidity_2m"
    )
    data = response.json()["hourly"]
    df_new = pd.DataFrame(data)
    df_spark_new = spark.createDataFrame(df_new)
    df_spark_new.write.format("delta").mode("append").save("Files/bronze/weather_raw")
    print(f"Adicionados {len(df_new)} registos históricos")

    # Forecast (sempre overwrite — são previsões futuras que mudam)
    response_forecast = requests.get(
        f"https://historical-forecast-api.open-meteo.com/v1/forecast?"
        f"latitude=40.66&longitude=-7.91"
        f"&start_date={today}&end_date={end_forecast}"
        f"&hourly=temperature_2m,relative_humidity_2m"
    )
    data_forecast = response_forecast.json()["hourly"]
    df_forecast = pd.DataFrame(data_forecast)
    df_spark_forecast = spark.createDataFrame(df_forecast)
    df_spark_forecast.write.format("delta").mode("overwrite").save("Files/bronze/weather_forecast")
    print(f"Forecast atualizado: {len(df_forecast)} registos")
else:
    print("Sem dados novos para processar")

StatementMeta(, 38404a5d-0d53-4e09-a5df-c6ec5cc64f2b, 3, Finished, Available, Finished, False)

Dados existentes até 2026-04-09, a pedir desde 2026-04-10
Adicionados 264 registos históricos
Forecast atualizado: 192 registos
